## PONDERACIÓN DE CARACTERÍSTICAS - WEIGHT MANAGER POR PASOS

PASO 1 - Inicializar con JSON de prueba (no el JSON que se usara)

In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from src.weighting.weight_manager import WeightManager

TEST_JSON = 'schemas_test/test_feature_weights.json'

print('=' * 60)
print('BLOQUE 1: INICIALIZACIÓN DESDE CONSTANTS (JSON DE PRUEBA)')
print('=' * 60)

wm = WeightManager(weight_path_=TEST_JSON)

print(f'\n  initialized_: {wm.initialized_}')
print(f'  weights_path_: {wm.weights_path_}')
print(f'  Total features cargadas: {len(wm.features_)}')
print(f'  Features: {wm.features_}')

BLOQUE 1: INICIALIZACIÓN DESDE CONSTANTS (JSON DE PRUEBA)
Pesos cargados desde schemas_test\test_feature_weights.json, total de características: 41.

  initialized_: True
  weights_path_: schemas_test\test_feature_weights.json
  Total features cargadas: 41
  Features: ['q2', 'edre', 'q10inc', 'etid', 'boletidnew', 'boletidnewb', 'ur', 'ocupoit', 'q1tc_r', 'q3cn', 'q5b', 'r3', 'r4a', 'r6', 'r7', 'r12', 'r15', 'r18n', 'r18', 'r16', 'r27', 'leng1', 'formal', 'wf1', 'bolcct1a', 'bolcct1b', 'bolcct1c', 'estratosec', 'q12cn', 'q12bn', 'wealth_index', 'gi0n', 'smedia3n', 'cp6', 'cp7', 'cp8', 'cp13', 'q10e', 'q11n', 'leng4', 'civic_index']


PASO 2 - Lectura de pesos

In [6]:
print('=' * 60)
print('BLOQUE 2: LECTURA DE PESOS')
print('=' * 60)

# Obtencion individual de pesos 
peso_q2 = wm.get_weight('q2')
print(f'\n  get_weight(q2): {peso_q2}')

peso_edre = wm.get_weight('edre')
print(f'  get_weight(edre): {peso_edre}')

# Obtención de odos los pesos
todos = wm.get_weights()
print(f'\n  get_weights() - primeros 5:')
for k, v in list(todos.items())[:5]:
    print(f'    {k:<20} {v}')

# Array ordenado
array = wm.get_weights_array()
print(f'\n  get_weights_array() - primeros 5: {array[:5]}')


BLOQUE 2: LECTURA DE PESOS

  get_weight(q2): 1.0
  get_weight(edre): 1.0

  get_weights() - primeros 5:
    q2                   1.0
    edre                 1.0
    q10inc               1.0
    etid                 0.95
    boletidnew           0.95

  get_weights_array() - primeros 5: [1.0, 1.0, 1.0, 0.95, 0.95]


PASO 3 - Modificación de pesos individuales

In [7]:
print('=' * 60)
print('BLOQUE 3: MODIFICACIÓN DE PESOS - set_weight')
print('=' * 60)

# Modificar un peso válido
print(f'\n  Peso edre antes: {wm.get_weight("edre")}')
wm.set_weight('edre', 0.85)
print(f'  Peso edre después de set_weight(edre, 0.85): {wm.get_weight("edre")}')

# Verificar que persiste con nueva instancia pero el mismo JSON
wm2 = WeightManager(weight_path_=TEST_JSON)
print(f'  Verificación de persistencia — nueva instancia de WeightManager get_weight(edre): {wm2.get_weight("edre")}')

# Prueba de validaciones
print('\n  --- Pruebas de validación ---')

try:
    wm.set_weight('feature_inexistente', 0.5)
except KeyError as e:
    print(f'  ✓ KeyError feature inexistente: {e}')

try:
    wm.set_weight('q2', 1.5)
except ValueError as e:
    print(f'  ✓ ValueError peso > 1.0: {e}')

try:
    wm.set_weight('q2', -0.1)
except ValueError as e:
    print(f'  ✓ ValueError peso < 0.0: {e}')

BLOQUE 3: MODIFICACIÓN DE PESOS - set_weight

  Peso edre antes: 1.0
  Peso edre después de set_weight(edre, 0.85): 0.85
Pesos cargados desde schemas_test\test_feature_weights.json, total de características: 41.
  Verificación de persistencia — nueva instancia de WeightManager get_weight(edre): 0.85

  --- Pruebas de validación ---
  ✓ KeyError feature inexistente: "Característica 'feature_inexistente' no encontrada en el archivo de pesos."
  ✓ ValueError peso > 1.0: Peso de 'q2' debe estar en [0.0, 1.0]. Recibido: 1.5
  ✓ ValueError peso < 0.0: Peso de 'q2' debe estar en [0.0, 1.0]. Recibido: -0.1


PASO 4 - Modificación de múltiples pesos

In [8]:
print('=' * 60)
print('BLOQUE 4: MODIFICACIÓN DE PESOS - set_weights')
print('=' * 60)

wm.reset_weights_values()  # Reiniciar a valores originales para esta prueba

nuevos_pesos = {
    'q2':        0.90,
    'edre':      0.95,
    'q10inc':    0.80,
    'etid':      0.75,
    'wealth_index': 0.70,
}

print(f'\n  Pesos antes:')
features = wm.get_weights()
for f in features:
    print(f'    {f:<20} {wm.get_weight(f)}')

wm.set_weights(nuevos_pesos)

features_nuevos_pesos = wm.get_weights()
print(f'\n  Pesos después de set_weights():')
for fnp in features_nuevos_pesos:
    print(f'    {fnp:<20} {wm.get_weight(fnp)}')

# Prueba de validación por lote
print('\n  --- Pruebas de validación ---')

try:
    wm.set_weights({'q2': 0.5, 'feature_falsa': 0.3})
except KeyError as e:
    print(f'  ✓ KeyError features desconocidas: {e}')

try:
    wm.set_weights({'q2': 0.5, 'edre': 2.0})
except ValueError as e:
    print(f'  ✓ ValueError peso inválido en lote: {e}')

BLOQUE 4: MODIFICACIÓN DE PESOS - set_weights
Pesos reiniciados a valores de constants.py.

  Pesos antes:
    q2                   1.0
    edre                 1.0
    q10inc               1.0
    etid                 0.95
    boletidnew           0.95
    boletidnewb          0.95
    ur                   0.9
    ocupoit              0.9
    q1tc_r               0.7
    q3cn                 0.7
    q5b                  0.7
    r3                   0.65
    r4a                  0.65
    r6                   0.65
    r7                   0.65
    r12                  0.65
    r15                  0.65
    r18n                 0.65
    r18                  0.65
    r16                  0.65
    r27                  0.65
    leng1                0.65
    formal               0.65
    wf1                  0.65
    bolcct1a             0.65
    bolcct1b             0.65
    bolcct1c             0.65
    estratosec           0.6
    q12cn                0.6
    q12bn                0.6
    

PASO 5 - Reset a valores de constants y pesos normalizados

In [9]:
print('=' * 60)
print('BLOQUE 5: RESET Y NORMALIZACIÓN')
print('=' * 60)

print(f'\n  Peso edre antes del reset: {wm.get_weight("edre")}')
print(f'  Peso q2 antes del reset: {wm.get_weight("q2")}')

wm.reset_weights_values()

print(f'\n  Peso edre después del reset: {wm.get_weight("edre")}')
print(f'  Peso q2 después del reset: {wm.get_weight("q2")}')

# Pesos normalizados (no modifica JSON)
print('\n  --- get_normalized_weights() ---')
originales = wm.get_weights()
normalizados = wm.get_normalized_weights()
print(f'  Suma de pesos originales: {sum(originales.values()):.6f} ')
print(f'  Suma de pesos normalizados: {sum(normalizados.values()):.6f} (debe ser 1.0)')
print(f'  Primeros 5 normalizados:')
for k, v in list(normalizados.items())[:5]:
    print(f'    {k:<20} {v}')

# Verificar JSON sin cambios
print(f'\n  Peso edre en JSON tras normalización: {wm.get_weight("edre")} (sin cambios)')

BLOQUE 5: RESET Y NORMALIZACIÓN

  Peso edre antes del reset: 0.95
  Peso q2 antes del reset: 0.9
Pesos reiniciados a valores de constants.py.

  Peso edre después del reset: 1.0
  Peso q2 después del reset: 1.0

  --- get_normalized_weights() ---
  Suma de pesos originales: 27.700000 
  Suma de pesos normalizados: 1.000011 (debe ser 1.0)
  Primeros 5 normalizados:
    q2                   0.036101
    edre                 0.036101
    q10inc               0.036101
    etid                 0.034296
    boletidnew           0.034296

  Peso edre en JSON tras normalización: 1.0 (sin cambios)


PASO 6 - Resumen visual

In [10]:

print('=' * 60)
print('BLOQUE 6: RESUMEN VISUAL')
print('=' * 60)

wm = WeightManager(weight_path_=TEST_JSON)

wm.print_summary()

BLOQUE 6: RESUMEN VISUAL
Pesos cargados desde schemas_test\test_feature_weights.json, total de características: 41.

  WeightManager — schemas_test\test_feature_weights.json
  Features: 41
  q2                   1.00  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  edre                 1.00  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  q10inc               1.00  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  etid                 0.95  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  boletidnew           0.95  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  boletidnewb          0.95  ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
  ur                   0.90  ■■■■■■■■■■■■■■■■■